# Model Evaluation for Prithvi EO Segmentation
This notebook demonstrates how to evaluate the Prithvi EO segmentation model using precision, recall, F1 score, and IoU metrics on validation data.

<a href="https://colab.research.google.com/github/easare377/Prithvi-EO-Segmentation/blob/main/model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install terratorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.5/153.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.6/323.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.0/211.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 851.6/851.6 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Install Dependencies and Mount Google Drive
Install required packages and mount Google Drive for accessing validation data and model weights.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import numpy as np
import pandas as pd
import torch

# Import Libraries and Define Dataset Class
Import necessary libraries and define the custom dataset class for loading TFRecord validation data.

In [2]:
import os, struct
import numpy as np
import tensorflow as tf
import torch
from torch.utils.data import Dataset
import random

def _zero_pad_hw(arr_hw_or_hwc, target_hw):
    th, tw = target_hw
    if arr_hw_or_hwc.ndim == 2:
        h, w = arr_hw_or_hwc.shape
        pad = ((max((th-h)//2,0), max(th-h,0) - max((th-h)//2,0)),
               (max((tw-w)//2,0), max(tw-w,0) - max((tw-w)//2,0)))
    else:
        h, w, c = arr_hw_or_hwc.shape
        pad = ((max((th-h)//2,0), max(th-h,0) - max((th-h)//2,0)),
               (max((tw-w)//2,0), max(tw-w,0) - max((tw-w)//2,0)),
               (0,0))
    return np.pad(arr_hw_or_hwc, pad, mode='constant')

def _zero_pad_cthw(X, target_hw):
    # pad only H,W for (C,T,H,W)
    C,T,H,W = X.shape
    img = np.transpose(X, (2,3,0,1)).reshape(H, W, C*T)      # (H,W,C*T)
    img = _zero_pad_hw(img, target_hw)
    H2, W2, CT = img.shape
    img = np.transpose(img, (2,0,1)).reshape(C, T, H2, W2)
    return img

class MineFootprintTFRecordDataset(Dataset):
    # Your 6-band stats; if C != 6 we fall back to per-sample normalization
    MEAN = np.array([1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0], dtype=np.float32)
    STD  = np.array([2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0], dtype=np.float32)

    _feature_desc = {
        "image_raw": tf.io.FixedLenFeature([], tf.string),
        "mask_raw":  tf.io.FixedLenFeature([], tf.string),
        "height":    tf.io.FixedLenFeature([], tf.int64),
        "width":     tf.io.FixedLenFeature([], tf.int64),
        "channels":  tf.io.FixedLenFeature([], tf.int64),
        "timesteps": tf.io.FixedLenFeature([], tf.int64),
        "temporal_coords": tf.io.VarLenFeature(tf.float32),  # length = 2*T
        "location_coords": tf.io.FixedLenFeature([2], tf.float32),
    }

    def __init__(self, tfrecord_file, transform=None, pad_to=(224, 224)):
        super().__init__()
        self.tfrecord_path = os.fspath(tfrecord_file)
        self.transform = transform
        self.pad_to = pad_to
        self._offsets = self._scan_index()
        self._fh = open(self.tfrecord_path, 'rb')

    def _scan_index(self):
        offsets = []
        with open(self.tfrecord_path, 'rb') as f:
            pos = 0
            while True:
                header = f.read(12)                  # 8 len + 4 len_crc
                if not header:
                    break
                rec_len = struct.unpack('<Q', header[:8])[0]
                offsets.append(pos)
                pos += 12 + rec_len + 4             # header + data + data_crc
                f.seek(pos)
        return offsets

    def _read_record(self, offset):
        self._fh.seek(offset)
        header = self._fh.read(12)
        rec_len = struct.unpack('<Q', header[:8])[0]
        data = self._fh.read(rec_len)
        _ = self._fh.read(4)
        return data

    def __len__(self):
        return len(self._offsets)

    def _normalize_cthw(self, X):
        C,T,H,W = X.shape
        if C == len(self.MEAN):
            mean = self.MEAN.reshape(C,1,1,1)
            std  = (self.STD + 1e-6).reshape(C,1,1,1)
            return (X - mean) / std
        # Fallback: per-sample per-channel across (T,H,W)
        m = X.mean(axis=(1,2,3), keepdims=True)
        s = X.std(axis=(1,2,3), keepdims=True) + 1e-6
        return (X - m) / s

    def _apply_albu_over_time(self, X, mask):
        """Apply Albumentations over (H,W) consistently for all time steps:
           reshape (C,T,H,W) -> (H,W,C*T), apply, then back."""
        C,T,H,W = X.shape
        img = np.transpose(X, (2,3,0,1)).reshape(H, W, C*T)   # (H,W,C*T)

        aug = self.transform(image=img, mask=mask)
        img_aug, msk_aug = aug["image"], aug["mask"]

        # Handle ToTensorV2 in pipeline
        if isinstance(img_aug, torch.Tensor):
            # Albumentations returns CHW if ToTensorV2; convert to HWC
            img_aug = img_aug.cpu().numpy().transpose(1,2,0)
        if isinstance(msk_aug, torch.Tensor):
            msk_aug = msk_aug.cpu().numpy()

        H2, W2, CT2 = img_aug.shape
        if CT2 % C != 0:
            raise RuntimeError(f"Aug channels {CT2} not divisible by C={C}")
        T2 = CT2 // C
        X2 = np.transpose(img_aug, (2,0,1)).reshape(C, T2, H2, W2)
        return X2, msk_aug

    def __getitem__(self, idx):
        serialised = self._read_record(self._offsets[idx])
        ex = tf.io.parse_single_example(serialised, self._feature_desc)

        H = int(ex["height"])
        W = int(ex["width"])
        C = int(ex["channels"])
        T = int(ex["timesteps"])

        img = np.frombuffer(ex["image_raw"].numpy(), dtype=np.float32).reshape((C, T, H, W))
        msk = np.frombuffer(ex["mask_raw"].numpy(),  dtype=np.uint8).reshape((H, W))

        img = np.nan_to_num(img, nan=0.0)
        msk = np.nan_to_num(msk.astype(np.float32), nan=0.0).astype(np.uint8)

        # Normalize per channel (broadcast over T,H,W)
        img = self._normalize_cthw(img)

        # Center-pad to target size
        img = _zero_pad_cthw(img, self.pad_to)       # (C,T,H2,W2)
        msk = _zero_pad_hw(msk, self.pad_to)         # (H2,W2)

        # Temporal/Location coords
        temporal_coords = tf.sparse.to_dense(ex["temporal_coords"]).numpy().astype(np.float32).reshape(T, 2)
        location_coords = ex["location_coords"].numpy().astype(np.float32)   # (2,)

        # Optional Albumentations over time (geom only)
        if self.transform is not None:
            img, msk = self._apply_albu_over_time(img, msk)

        # To torch
        img = torch.from_numpy(img).float()                  # (C,T,H,W)
        msk = torch.from_numpy(msk).long()                   # (H,W)
        temporal_coords = torch.from_numpy(temporal_coords)  # (T,2)
        location_coords = torch.from_numpy(location_coords)  # (2,)

        return {
            "image": img,  # (C,T,H,W)
            "temporal_coords": temporal_coords,  # (T,2)
            "location_coords": location_coords,  # (2,)
            "mask": msk      # (H,W)
        }

    def __del__(self):
        try:
            if hasattr(self, "_fh") and self._fh and not self._fh.closed:
                self._fh.close()
        except Exception:
            pass

# -------- collate: pick ONE random timestep length for the whole batch --------
def make_temporal_collate(max_T=10, min_T=1):
    assert 1 <= min_T <= max_T
    def _collate(batch):
        t_sel = 4 #random.randint(min_T, max_T)
        imgs, tcoords, lcoords, masks = [], [], [], []
        for sample in batch:
            X = sample["image"]                 # (C,T,H,W)
            T = X.shape[1]
            X = X[:, -t_sel:, :, :]             # keep LAST t_sel
            imgs.append(X)

            tc = sample["temporal_coords"]      # (T,2)
            tcoords.append(tc[-t_sel:, :])

            lcoords.append(sample["location_coords"])   # (2,)
            masks.append(sample["mask"])         # (H,W)

        imgs   = torch.stack(imgs,   dim=0)      # (B,C,t_sel,H,W)
        tcoords= torch.stack(tcoords,dim=0)      # (B,t_sel,2)
        lcoords= torch.stack(lcoords,dim=0)      # (B,2)
        masks  = torch.stack(masks,  dim=0)      # (B,H,W)
        return {"image": imgs,
                "temporal_coords": tcoords,
                "location_coords": lcoords,
                "mask": masks
               }
    return _collate

def make_temporal_collate(timesteps=4):
    #assert 1 <= min_T <= max_T
    def _collate(batch):
        t_sel = timesteps #random.randint(min_T, max_T)
        imgs, tcoords, lcoords, masks = [], [], [], []
        for sample in batch:
            X = sample["image"]                 # (C,T,H,W)
            T = X.shape[1]
            X = X[:, -t_sel:, :, :]             # keep LAST t_sel
            imgs.append(X)

            tc = sample["temporal_coords"]      # (T,2)
            tcoords.append(tc[-t_sel:, :])

            lcoords.append(sample["location_coords"])   # (2,)
            masks.append(sample["mask"])         # (H,W)

        imgs   = torch.stack(imgs,   dim=0)      # (B,C,t_sel,H,W)
        tcoords= torch.stack(tcoords,dim=0)      # (B,t_sel,2)
        lcoords= torch.stack(lcoords,dim=0)      # (B,2)
        masks  = torch.stack(masks,  dim=0)      # (B,H,W)
        return {"image": imgs,
                "temporal_coords": tcoords,
                "location_coords": lcoords,
                "mask": masks
               }
    return _collate

In [3]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
# Define transformation pipeline
transform = A.Compose([
    A.RandomRotate90(p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    ToTensorV2(),
])

In [14]:
val_file_path = r'D:\ssm_footprint_val.tfrecord'
TIMESTEPS = 4
val_dataset = MineFootprintTFRecordDataset(val_file_path, transform=transform)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=2, shuffle=False, drop_last=True, collate_fn=make_temporal_collate(TIMESTEPS))

In [16]:
train_file_path = r'D:\ssm_footprint_train.tfrecord'
TIMESTEPS = 4
train_dataset = MineFootprintTFRecordDataset(train_file_path, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=2, shuffle=True, drop_last=True, collate_fn=make_temporal_collate(TIMESTEPS))

# Prepare Validation Data Loader
Load the validation TFRecord file and prepare the DataLoader for evaluation.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# assumes PrithviViT and FCNDecoder are imported from your source above
# from your_module import PrithviViT, FCNDecoder, get_3d_sincos_pos_embed
from terratorch.models.backbones.prithvi_mae import PrithviViT, get_3d_sincos_pos_embed
from terratorch.models.decoders import FCNDecoder


# ---------- helpers -----------------------------------------------------------
def set_temporal_frames(encoder: PrithviViT, T: int):
    """Make the encoder expect T frames and rebuild its pos_embed on the right device/dtype."""
    ps_t, ps_h, ps_w = encoder.patch_embed.patch_size
    assert ps_t == 1, f"temporal patch must be 1, got {ps_t}"
    encoder.num_frames = T
    encoder.patch_embed.input_size = (T,) + encoder.img_size
    encoder.patch_embed.grid_size  = [s // p for s, p in zip(encoder.patch_embed.input_size,
                                                             encoder.patch_embed.patch_size)]

    pos = get_3d_sincos_pos_embed(encoder.embed_dim, encoder.patch_embed.grid_size, add_cls_token=True)
    pos = torch.from_numpy(pos).float().unsqueeze(0).to(
        encoder.cls_token.device, dtype=encoder.cls_token.dtype
    )
    # replace buffer (non-persistent so it follows device moves cleanly)
    encoder.register_buffer("pos_embed", pos, persistent=False)


def patch_pos_interp_device_safety(encoder: PrithviViT):
    """
    Some builds return CPU tensors from interpolate_pos_encoding; make sure they land on the buffer's device/dtype.
    """
    _old = encoder.interpolate_pos_encoding

    def _safe(sample_shape):
        out = _old(sample_shape)
        return out.to(encoder.pos_embed.device, dtype=encoder.pos_embed.dtype)

    encoder.interpolate_pos_encoding = _safe  # bind to instance


# ---------- model -------------------------------------------------------------
class PrithviFCNSeg(nn.Module):
    """
    PrithviViT encoder -> token-to-image reshape -> FCNDecoder -> 1x1 head.

    Forward expects:
        x: (B, C, T, H, W) or (B, C, H, W) if T==1
        temporal_coords: (B, T, 2) or None
        location_coords: (B, 2) or None
    Returns:
        logits: (B, num_classes, H, W)
    """
    def __init__(
        self,
        encoder: PrithviViT,
        decoder: FCNDecoder,
        num_classes: int,
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.head    = nn.Conv2d(decoder.out_channels, num_classes, kernel_size=1)

    def forward(self, x, temporal_coords=None, location_coords=None):
        B = x.shape[0]
        H, W = x.shape[-2], x.shape[-1]

        # 1) ViT features (list of token sequences, each with CLS token)
        feats = self.encoder.forward_features(
            x, temporal_coords=temporal_coords, location_coords=location_coords
        )  # list[Tensor], each [B, 1 + L, embed_dim]

        # 2) Minimal "neck": reshape tokens to (B, C*T, h, w) per level
        maps = self.encoder.prepare_features_for_image_model(feats)  # list[Tensor], e.g. last is [B, E*T, 14, 14]

        # 3) FCN decoder (expects list, picks in_index in its own forward)
        decoded = self.decoder(maps)               # (B, decoder_channels, Hdec, Wdec)

        # 4) 1x1 head to logits
        logits = self.head(decoded)                # (B, num_classes, Hdec, Wdec)

        # 5) Upsample back to input spatial size if needed
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)

        return logits


# ---------- factory -----------------------------------------------------------
def build_prithvi_fcn_segmentation(
    *,
    num_classes: int = 3,
    in_chans: int = 6,
    img_size: int = 224,
    T: int = 10,
    patch_size=(1, 16, 16),
    embed_dim: int = 1024,
    depth: int = 24,
    num_heads: int = 16,
    mlp_ratio: float = 4.0,
    coords_encoding=("time", "location"),
    decoder_channels: int = 256,
    decoder_num_convs: int = 4,
    device: torch.device | str = "cuda",
):
    # 1) Encoder
    encoder = PrithviViT(
        img_size=img_size,
        patch_size=patch_size,  # (1,16,16) required for temporal encoding
        num_frames=T,
        in_chans=in_chans,
        embed_dim=embed_dim,
        depth=depth,
        num_heads=num_heads,
        mlp_ratio=mlp_ratio,
        coords_encoding=list(coords_encoding),
    )

    # Make pos-embed interpolation device-safe and lock T
    patch_pos_interp_device_safety(encoder)
    set_temporal_frames(encoder, T)

    # 2) Decoder: expects a list of embed_dims per stage; encoder.out_channels is already that list
    decoder = FCNDecoder(embed_dim=encoder.out_channels, channels=decoder_channels, num_convs=decoder_num_convs)

    # 3) Compose full model
    model = PrithviFCNSeg(encoder, decoder, num_classes=num_classes)
    return model.to(device)


c:\Users\emmanuelasare\AppData\Local\anaconda3\envs\ml-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_prithvi_fcn_segmentation(
    num_classes=3,
    in_chans=6,
    img_size=224,
    T=TIMESTEPS,
    device=device,
)

B, C, T, H, W = 2, 6, TIMESTEPS, 224, 224
x = torch.randn(B, C, T, H, W, device=device)

years = torch.arange(2015, 2015+T, device=device).float().unsqueeze(0).repeat(B, 1)
tcoords = torch.stack([years, torch.ones_like(years)], dim=-1)      # (B,T,2)
lcoords = torch.tensor([[7.12, -1.5]] * B, device=device).float()   # (B,2)

with torch.no_grad():
    logits = model(x, temporal_coords=tcoords, location_coords=lcoords)
print("logits:", tuple(logits.shape))  # -> (B, num_classes, H, W)

logits: (2, 3, 224, 224)


# Load and Prepare Segmentation Model
Instantiate the Prithvi EO segmentation model and load pretrained weights for evaluation.

In [7]:
import os
import torch
import torch.nn.functional as F

@torch.no_grad()
def load_prithvi_encoder_weights_for_seg(
    seg_model,                 # instance of PrithviFCNSeg from earlier
    ckpt_path: str,
    *,
    strict: bool = False,
    resize_patch_if_needed: bool = False,
):
    """
    Load pretrained Prithvi encoder weights into seg_model.encoder.

    - Accepts checkpoints that store either a plain state_dict or {"state_dict": ...}.
    - Searches common key layouts and remaps them onto seg_model.encoder.
    - Drops pos_embed if its shape doesn't match current encoder.pos_embed.
    - (Optional) Resizes patch_embed.proj.weight (3D conv) if kernel shape differs.

    Args:
        seg_model: your PrithviFCNSeg instance
        ckpt_path: path to checkpoint file
        strict: pass True if you expect an exact structural match
        resize_patch_if_needed: if True, resizes temporal/spatial kernel dims to match your model
    """
    assert os.path.isfile(ckpt_path), f"Checkpoint not found: {ckpt_path}"
    device = next(seg_model.parameters()).device

    sd = torch.load(ckpt_path, map_location="cpu")
    state = sd.get("state_dict", sd)

    enc = seg_model.encoder
    model_sd = enc.state_dict()

    # -------- collect encoder weights under common prefixes ----------
    enc_sd = {}
    for k, v in state.items():
        # common layouts
        if k.startswith("model.encoder."):
            enc_sd[k.replace("model.encoder.", "")] = v
        elif k.startswith("encoder."):
            enc_sd[k.replace("encoder.", "")] = v
        elif k.startswith("model.backbone.encoder."):
            enc_sd[k.replace("model.backbone.encoder.", "")] = v
        elif k.startswith("backbone.encoder."):
            enc_sd[k.replace("backbone.encoder.", "")] = v
        # fallback: exact match into encoder keys
        elif k in model_sd:
            enc_sd[k] = v

    # -------- safety: drop/adjust incompatible shapes ----------------
    # 1) pos_embed shape must match tokens (depends on T, H/patch, W/patch)
    pe = enc_sd.get("pos_embed", None)
    if isinstance(pe, torch.Tensor) and pe.shape != enc.pos_embed.shape:
        enc_sd.pop("pos_embed", None)
        print(f"[weights] dropped pos_embed (ckpt {tuple(pe.shape)} != model {tuple(enc.pos_embed.shape)})")

    # 2) Optional: resize patch_embed.proj.weight (out, in, kt, kh, kw)
    if resize_patch_if_needed:
        tgt_w = enc.patch_embed.proj.weight.shape
        ck_w = enc_sd.get("patch_embed.proj.weight", None)
        if isinstance(ck_w, torch.Tensor) and ck_w.shape != tgt_w:
            # reshape to merge (out*in) for clean 3D interpolate
            oi = ck_w.shape[0] * ck_w.shape[1]
            w_oi = ck_w.reshape(oi, 1, ck_w.shape[2], ck_w.shape[3], ck_w.shape[4]).float()
            w_resized = F.interpolate(
                w_oi, size=(tgt_w[2], tgt_w[3], tgt_w[4]),
                mode="trilinear", align_corners=True
            ).reshape(tgt_w).to(dtype=ck_w.dtype)
            enc_sd["patch_embed.proj.weight"] = w_resized
            print(f"[weights] resized patch_embed.proj.weight {tuple(ck_w.shape)} -> {tuple(tgt_w)}")

        # bias can also mismatch → drop it
        ck_b = enc_sd.get("patch_embed.proj.bias", None)
        if isinstance(ck_b, torch.Tensor) and ck_b.shape != enc.patch_embed.proj.bias.shape:
            enc_sd.pop("patch_embed.proj.bias", None)
            print("[weights] dropped patch_embed.proj.bias (shape mismatch)")

    # 3) Drop any remaining mismatches
    dropped = []
    for k, v in list(enc_sd.items()):
        if k in model_sd and model_sd[k].shape != v.shape:
            dropped.append((k, tuple(v.shape), tuple(model_sd[k].shape)))
            enc_sd.pop(k, None)
    if dropped:
        print("[weights] dropped for shape mismatch:", dropped[:6], "..." if len(dropped) > 6 else "")

    # -------- load ----------------------------------------------------
    missing, unexpected = enc.load_state_dict(enc_sd, strict=strict)
    # move buffers to model device/dtype (pos_embed already correct; just ensure device)
    seg_model.to(device)

    print(f"[weights] encoder loaded. missing={len(missing)} unexpected={len(unexpected)} (strict={strict})")
    if missing:    print("  missing:", missing[:10], "..." if len(missing) > 10 else "")
    if unexpected: print("  unexpected:", unexpected[:10], "..." if len(unexpected) > 10 else "")


In [8]:
load_prithvi_encoder_weights_for_seg(
    model,
    r"prithvi_state_dict.pt",
    strict=False,
    resize_patch_if_needed=False,   # set True only if your patch size differs from the ckpt
)

[weights] encoder loaded. missing=0 unexpected=0 (strict=False)


In [9]:
model.load_state_dict(torch.load('prithvi_state_dict.pt'))

<All keys matched successfully>

In [10]:
def calculate_precision_recall(ground_truth_mask, predicted_mask, num_classes):
    # print(ground_truth_mask.shape, predicted_mask.shape)
    precision = [None] * num_classes  # np.zeros(num_classes)
    recall = [None] * num_classes  # np.zeros(num_classes)
    precision_dict = dict()
    recall_dict = dict()
    for class_id in range(num_classes):
        true_positives = np.sum(
            (ground_truth_mask == class_id) & (predicted_mask == class_id))
        false_positives = np.sum(
            (ground_truth_mask != class_id) & (predicted_mask == class_id))
        false_negatives = np.sum(
            (ground_truth_mask == class_id) & (predicted_mask != class_id))
        p = None
        r = None
        if true_positives + false_positives != 0:
            p = true_positives / (true_positives + false_positives)
        if true_positives + false_negatives != 0:
            r = true_positives / (true_positives + false_negatives)
        precision_dict[class_id] = p
        recall_dict[class_id] = r
    return precision_dict, recall_dict


def calc_f1_score(precision, recall):
    if precision is None or recall is None or (precision + recall) == 0:
        return None
    return (2 * precision * recall) / (precision + recall)


def calculate_raw_scores(ground_truth_mask, predicted_mask, num_classes):
    # print(ground_truth_mask.shape, predicted_mask.shape)
    # precision = [None] * num_classes  # np.zeros(num_classes)
    # recall = [None] * num_classes  # np.zeros(num_classes)
    # precision_dict = dict()
    # recall_dict = dict()
    true_positives_dict = dict()
    false_positives_dict = dict()
    false_negatives_dict = dict()
    for class_id in range(num_classes):
        true_positives = np.sum(
            (ground_truth_mask == class_id) & (predicted_mask == class_id))
        false_positives = np.sum(
            (ground_truth_mask != class_id) & (predicted_mask == class_id))
        false_negatives = np.sum(
            (ground_truth_mask == class_id) & (predicted_mask != class_id))
        true_positives_dict[class_id] = true_positives
        false_positives_dict[class_id] = false_positives
        false_negatives_dict[class_id] = false_negatives
        # p = None
        # r = None
        # if true_positives + false_positives != 0:
        #     p = true_positives / (true_positives + false_positives)
        # if true_positives + false_negatives != 0:
        #     r = true_positives / (true_positives + false_negatives)
        # precision_dict[class_id] = p
        # recall_dict[class_id] = r
    return true_positives_dict, false_positives_dict, false_negatives_dict

def calculate_mean_precision_recall(model, val_datagen, num_classes):
    # Replace with the actual number of classes in your model
    # num_classes = 3
    # precision_total_dict = dict()
    # recall_total_dict = dict()
    total_true_positives_dict = dict()
    total_false_positives_dict = dict()
    total_false_negatives_dict = dict()
    precision_scores = np.zeros(num_classes)
    recall_scores = np.zeros(num_classes)
    f1_scores = np.zeros(num_classes)
    # class_iou_scores = np.zeros(num_classes)
    # class_counts = np.zeros(num_classes)

    # Iterate over the validation data generator
    for images, labels in val_datagen:
        for image, label in zip(images, labels):
            # Predict the labels using the model
            predictions = model.predict(np.array([image]), verbose=False)
            predicted_classes = np.argmax(predictions, axis=3)
            # print(predicted_classes.shape)
            true_positives_dict, false_positives_dict, false_negatives_dict = calculate_raw_scores(
                label, predicted_classes[0], num_classes)
            # print(precision_dict, recall_dict)
            for class_id in range(num_classes):
                # p = precision_dict.get(class_id)
                # r = recall_dict.get(class_id)
                ttp = total_true_positives_dict.get(class_id, 0.0)
                tfp = total_false_positives_dict.get(class_id, 0.0)
                tfn = total_false_negatives_dict.get(class_id, 0.0)
                ttp += true_positives_dict[class_id]
                tfp += false_positives_dict[class_id]
                tfn += false_negatives_dict[class_id]
                total_true_positives_dict[class_id] = ttp
                total_false_positives_dict[class_id] = tfp
                total_false_negatives_dict[class_id] = tfn
                # if p != None:
                #     pt = precision_total_dict.get(class_id, [0, 0])
                #     pt[0] += p
                #     pt[1] += 1
                #     precision_total_dict[class_id] = pt
                # if r != None:
                #     rt = recall_total_dict.get(class_id, [0, 0])
                #     rt[0] += r
                #     rt[1] += 1
                #     recall_total_dict[class_id] = rt
        # Compute IoU scores for each class
        # for class_id in range(num_classes):

    # Calculate average IoU scores for each class
    # class_iou_scores /= class_counts
    for class_id in range(num_classes):
        ttp = total_true_positives_dict.get(class_id)
        tfp = total_false_positives_dict.get(class_id)
        tfn = total_false_negatives_dict.get(class_id)
        p = None
        r = None
        if ttp + tfp != 0:
            p = ttp / (ttp + tfp)
        if ttp + tfn != 0:
            r = ttp / (ttp + tfn)
        # pt = precision_total_dict.get(x)
        # rt = recall_total_dict.get(x)
        # total_score = pt[0]
        precision_scores[class_id] = p
        # rt = recall_total_dict.get(x)
        # total_score = pt[0]
        recall_scores[class_id] = r
        f1_scores[class_id] = calc_f1_score(p, r)

    # Create a DataFrame with class labels and corresponding IoU scores
    iou_df = pd.DataFrame(
        {'class_val': range(num_classes), 'precision': precision_scores, 'recall': recall_scores, 'f1 score': f1_scores})
    # print(precision_total_dict)
    return iou_df


def calculate_mean_precision_recall(model_func, model, val_datagen, num_classes, device):
    total_true_positives_dict = {c: 0 for c in range(num_classes)}
    total_false_positives_dict = {c: 0 for c in range(num_classes)}
    total_false_negatives_dict = {c: 0 for c in range(num_classes)}

    # For batchwise prediction, create predictor
    predict = model_func(model, device)

    for batch in val_datagen:
        # images: batch of shape [B, C, H, W] (torch.Tensor or np.ndarray)
        # labels: batch of shape [B, H, W] (numpy or torch)
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        temporal_coords=batch["temporal_coords"].to(device)
        location_coords=batch["location_coords"].to(device)
        # if torch.is_tensor(images):
        #     images = images.to(device)
        pred_masks = predict(images, temporal_coords=temporal_coords, location_coords=location_coords)  # shape: [B, H, W] (np.ndarray)
        if torch.is_tensor(masks):
            masks = masks.cpu().numpy()
        for pred_mask, label in zip(pred_masks, masks):
            true_positives_dict, false_positives_dict, false_negatives_dict = calculate_raw_scores(
                label, pred_mask, num_classes)
            for class_id in range(num_classes):
                total_true_positives_dict[class_id] += true_positives_dict[class_id]
                total_false_positives_dict[class_id] += false_positives_dict[class_id]
                total_false_negatives_dict[class_id] += false_negatives_dict[class_id]

    # Aggregate results
    precision_scores = np.zeros(num_classes)
    recall_scores = np.zeros(num_classes)
    f1_scores = np.zeros(num_classes)
    iou_scores = np.zeros(num_classes)

    for class_id in range(num_classes):
        ttp = total_true_positives_dict[class_id]
        tfp = total_false_positives_dict[class_id]
        tfn = total_false_negatives_dict[class_id]
        denom = ttp + tfp + tfn
        iou = ttp / denom if denom != 0 else None
        p = ttp / (ttp + tfp) if (ttp + tfp) != 0 else None
        r = ttp / (ttp + tfn) if (ttp + tfn) != 0 else None
        f1 = calc_f1_score(p, r)
        precision_scores[class_id] = p if p is not None else np.nan
        recall_scores[class_id] = r if r is not None else np.nan
        f1_scores[class_id] = f1 if f1 is not None else np.nan
        iou_scores[class_id] = iou if iou is not None else np.nan

    iou_df = pd.DataFrame({
        'class_val': range(num_classes),
        'iou': iou_scores,
        'precision': precision_scores,
        'recall': recall_scores,
        'f1 score': f1_scores
    })
    return iou_df




# Define Evaluation Metrics and Functions
Implement functions to calculate precision, recall, F1 score, and IoU for model predictions.

In [11]:
import torch
import numpy as np
import pandas as pd

def model_predict(model, device):
    """
    Returns a function that predicts class masks from input tensors using a model.
    Handles both single image and batch input.
    """
    def predict(images, temporal_coords, location_coords):
        model.to(device)
        model.eval()
        with torch.no_grad():
            if isinstance(images, np.ndarray):
                images = torch.from_numpy(images)
            images = images.to(device)
            # if images.dim() == 3:  # Single image, shape: [C, H, W]
            #     images = images.unsqueeze(0)
            outputs = model(images)  # [B, num_classes, H, W]
            pred_masks = torch.argmax(outputs, dim=1)  # [B, H, W]
            pred_masks = pred_masks.cpu().numpy()
        return pred_masks
    return predict


In [12]:
def model_predict(model, device):
    """
    Returns a function that predicts class masks from input tensors using a model.
    Handles both single image and batch input.
    Supports optional temporal_coords and location_coords.
    """
    def predict(images, temporal_coords=None, location_coords=None):
        model.to(device)
        model.eval()
        with torch.no_grad():
            if isinstance(images, np.ndarray):
                images = torch.from_numpy(images)
            images = images.to(device)
            if images.dim() == 3:  # Single image, shape: [C, H, W]
                images = images.unsqueeze(0)

            # Prepare optional kwargs
            kwargs = {}
            if temporal_coords is not None:
                if isinstance(temporal_coords, np.ndarray):
                    temporal_coords = torch.from_numpy(temporal_coords)
                kwargs["temporal_coords"] = temporal_coords.to(device)
            if location_coords is not None:
                if isinstance(location_coords, np.ndarray):
                    location_coords = torch.from_numpy(location_coords)
                kwargs["location_coords"] = location_coords.to(device)

            outputs = model(images, **kwargs)#.output  # [B, num_classes, H, W]
            pred_masks = torch.argmax(outputs, dim=1)  # [B, H, W]
            pred_masks = pred_masks.cpu().numpy()
        return pred_masks
    return predict


# Define Model Prediction Helper Functions
Create helper functions to run model inference and obtain predicted masks for evaluation.

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [17]:
results_df = calculate_mean_precision_recall(model_predict, model, train_loader, 3, device)
results_df

,class_val,iou,precision,recall,f1 score
0,0,0.991825,0.996754,0.995039,0.995896
1,1,0.644408,0.758639,0.810596,0.783757
2,2,0.525714,0.593679,0.821179,0.689139


In [15]:
results_df = calculate_mean_precision_recall(model_predict, model, val_loader, 3, device)
results_df

,class_val,iou,precision,recall,f1 score
0,0,0.992590,0.993579,0.998999,0.996281
1,1,0.340757,0.814850,0.369355,0.508305
2,2,0.549875,0.616347,0.836029,0.709574


# Run Evaluation and Display Results
Run the evaluation metrics on the validation set and display precision, recall, F1 score, and IoU for each class.